# Module 1 — Homework 1 (2026 cohort)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jschuller/stock-markets-analytics-zoomcamp/blob/main/my-notes/01-intro/homework1.ipynb)

**Due 2026-09-14, 22:59** · questions in [`cohorts/2026/homework1.md`](../../cohorts/2026/homework1.md)
· [submission form](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01)
· [leaderboard](https://courses.datatalks.club/sma-zoomcamp-2026/leaderboard)

Four scored questions plus two optional free-text ones. Each scored question gets
its own cell and prints its value in an `ANSWER —` block.

Runs unchanged in **three environments** — local Jupyter, Google Colab, and a
Databricks notebook. The setup cell detects which one it is in and installs only
what is missing; nothing else in the notebook is environment-specific.

## Reproducibility

The lecture notebook derives its window from `date.today()` minus 70 years, so its
numbers change every run. A homework answer cannot do that, so **every window here
is pinned to a constant** declared in the setup cell. Re-running this notebook in
December must print what it printed today.

## Two ambiguities, since resolved upstream

The question sheet originally contradicted itself twice. Both were corrected on
2026-08-24, hours after it was published, and both landed on the reading this
notebook had already taken:

- **Q2** was titled "as of 21 August 2026" and hinted `end_date='2026-08-21'` while
  its prose said "1 January-1 August 2026". Commit `a6987b0` changed the prose to
  21 August. The 08-01 figure is still printed, now as a sensitivity check.
- **Q4**'s heading asked for the *median 2-day change after positive surprises* while
  step 4 asked for *the correlation*. Commit `a5cf53a` rewrote step 4 to ask for the
  median **first** and the correlation second. The median is the submitted answer.
- **Q3** defined a correction as a fall of "more than 5%" while its own step 5 said
  "at least 5%". Commit `a5cf53a` settled on **at least**, which is the `>=` this
  notebook was already using.

## Known traps in this material

- `pd.read_html` on Wikipedia is **HTTP 403** without a browser `User-Agent` — the
  question's own hint says so.
- yfinance wants `%Y-%m-%d` date bounds. Passing `"2026-08-22 00:00:00"` fails with
  `ValueError: unconverted data remains: 00:00:00`, which surfaces as an **empty frame
  for every ticker**, not as an error.
- `get_earnings_dates()` returns a **tz-aware** index (America/New_York) at 16:00, i.e.
  after the close. Comparing it to naive timestamps raises.
- yfinance 404s some live symbols. As of 2026-08-24 `MMC`, `FI` and `BK` all fail at
  Yahoo's own chart endpoint — check there before debugging client code.
- 2025's notebook read `^SPX` from Stooq, which returned rows **reversed**, so it used
  negative shifts. Yahoo is chronological — do not copy that sign convention.

## Setup

In [1]:
import importlib.util
import io
import os
import subprocess
import sys
import datetime as dt


def detect_env():
    """local | colab | databricks — decided once, used by the installer below."""
    if "google.colab" in sys.modules:
        return "colab"
    try:
        dbutils          # noqa: F821 — Databricks injects this into notebook globals
        return "databricks"
    except NameError:
        pass
    if os.environ.get("DATABRICKS_RUNTIME_VERSION"):
        return "databricks"       # classic clusters set this; serverless may not
    return "local"


def ensure(*packages):
    """Install only what is actually missing.

    Deliberately subprocess and not `!pip` or `%pip`. `%pip` aborts the entire run
    on failure with CalledProcessError, and neither magic survives headless
    execution by nbconvert or the Databricks Jobs API — both of which this notebook
    is run under. Same idiom as my-notes/databricks/bundle/src/01_env_probe.py.
    """
    missing = [p for p in packages
               if importlib.util.find_spec(p.split("==")[0].replace("-", "_")) is None]
    if missing:
        print(f"installing: {', '.join(missing)}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                       check=True)


ENV = detect_env()
# Colab ships numpy/pandas/requests/lxml but not yfinance; Databricks serverless
# ships none of them; a local conda env built from my-notes/environment.yml has
# them all, so this is a no-op there.
ensure("yfinance")

import numpy as np
import pandas as pd
import requests

import yfinance as yf

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

# ---------------------------------------------------------------------------
# THE PINS. Everything below derives from these — never from date.today().
# ---------------------------------------------------------------------------
HISTORY_START = dt.date(1950, 1, 1)     # Q3 asks for 1950-present

Q1_MIN_YEAR = 2020                      # "STARTING FROM 2020"
Q2_START    = dt.date(2026, 1, 1)       # hint: start_date='2026-01-01'
Q2_END      = dt.date(2026, 8, 21)      # title + hint; prose says 08-01, see below
Q2_END_ALT  = dt.date(2026, 8, 1)       # the prose reading, printed for comparison
Q4_TICKER   = "AMZN"

ANSWERS = {}        # collected by answer(); emitted as JSON by the final cell

print(f"environment: {ENV}")
print(f"yfinance {yf.__version__} | pandas {pd.__version__} | python {sys.version.split()[0]}")
print(f"Q2 window pinned to {Q2_START} -> {Q2_END}")

environment: local
yfinance 1.6.0 | pandas 2.3.3 | python 3.11.16
Q2 window pinned to 2026-01-01 -> 2026-08-21


### Helpers

In [2]:
BROWSER_UA = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                            "AppleWebKit/537.36 (KHTML, like Gecko) "
                            "Chrome/91.0.4472.124 Safari/537.36")}


def fetch_tables(url, headers=None, timeout=30):
    """Return every HTML table at `url` as a DataFrame.

    pd.read_html(url) does its own fetch with a urllib User-Agent, which Wikipedia
    answers with 403. Fetching via requests with a browser UA and handing read_html
    the *text* is the way round it — the question's own hint shows the same shape.
    """
    r = requests.get(url, headers=headers or BROWSER_UA, timeout=timeout)
    r.raise_for_status()
    return pd.read_html(io.StringIO(r.text))


def get_ohlcv(tickers, start, end=None, auto_adjust=False):
    """yfinance -> {ticker: DataFrame} with a tz-naive DatetimeIndex.

    Normalises the one-vs-many ticker column shape, which is the usual source of
    silent bugs when a question moves from one ticker to eleven.

    Both bounds are formatted %Y-%m-%d. yfinance parses dates with an exact format
    and rejects the "2026-08-22 00:00:00" that str(pd.Timestamp(...)) produces, with
    ValueError("unconverted data remains: 00:00:00") — which shows up as an empty
    frame for every ticker rather than as an obvious failure.

    `end` is exclusive in yfinance, so it is bumped one day to make the homework's
    inclusive "as of <date>" wording behave as written.
    """
    single = isinstance(tickers, str)
    tickers = [tickers] if single else list(tickers)
    d8 = lambda x: pd.Timestamp(x).strftime("%Y-%m-%d")

    kw = dict(start=d8(start), auto_adjust=auto_adjust, actions=False,
              progress=False, group_by="ticker")
    if end is not None:
        kw["end"] = d8(pd.Timestamp(end) + pd.Timedelta(days=1))

    raw = yf.download(tickers, **kw)
    out = {}
    for t in tickers:
        if isinstance(raw.columns, pd.MultiIndex):
            if t not in raw.columns.get_level_values(0):
                continue
            df = raw[t].copy()
        else:
            df = raw.copy()
        df = df.dropna(subset=["Close"])
        if df.empty:
            continue
        if getattr(df.index, "tz", None) is not None:
            df.index = df.index.tz_localize(None)
        out[t] = df

    missing = [t for t in tickers if t not in out]
    if missing:
        print(f"  !! no data returned for: {missing}")
    return out


def answer(key, label, value, note=""):
    """Print a submitted value unmissably, and record it for the final JSON block.

    The homework is graded on these scalars, so they are worth making impossible
    to miss in a long notebook.
    """
    ANSWERS[key] = value
    bar = "=" * 64
    print(f"\n{bar}\nANSWER — {label}\n  >>> {value} <<<")
    if note:
        print(f"  {note}")
    print(bar)
    return value

---
## Q1 — S&P 500 additions by year

> Which (full) year had the highest number of additions, **starting from 2020**?

Note the difference from 2025, which excluded only 1957 and searched all history.
The 2020 floor also makes the "full year" wording matter: 2026 is still in progress,
so it is excluded — a partial year cannot win a "highest number of additions" contest
on equal terms.

In [3]:
Q1_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

tables = fetch_tables(Q1_URL)
print(f"tables found: {len(tables)} | shapes: {[t.shape for t in tables[:3]]}")

constituents = tables[0]
constituents.columns = [str(c).strip() for c in constituents.columns]
print(f"columns: {list(constituents.columns)}")
constituents.head(3)

tables found: 2 | shapes: [(503, 8), (11, 2)]
columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888


In [4]:
# The column has been called "Date added" and "Date first added" across revisions,
# and values range from "1976-08-09" to "1957" to "March 4, 1957". Resolve the name,
# then pull a 4-digit year rather than trusting a date parser.
date_col = next(c for c in constituents.columns
                if "date" in c.lower() and "added" in c.lower())
tick_col = next(c for c in constituents.columns
                if "symbol" in c.lower() or "ticker" in c.lower())
print(f"using date column {date_col!r}, ticker column {tick_col!r}")

q1 = constituents[[tick_col, date_col]].copy()
q1.columns = ["ticker", "date_added"]
q1["year_added"] = (q1["date_added"].astype(str)
                    .str.extract(r"(\d{4})")[0].astype("Int64"))

print(f"\nconstituents: {len(q1)} | year parsed for {q1['year_added'].notna().sum()}")

per_year = (q1.dropna(subset=["year_added"])
              .groupby("year_added").size().sort_index())

# "Full year", so the in-progress current year cannot compete on equal footing.
CURRENT_YEAR = int(per_year.index.max())
full_years = per_year[(per_year.index >= Q1_MIN_YEAR) & (per_year.index < CURRENT_YEAR)]

print(f"\nadditions per year from {Q1_MIN_YEAR} "
      f"({CURRENT_YEAR} excluded as incomplete — it has {per_year.get(CURRENT_YEAR, 0)} so far):")
print(full_years.to_string())

best = int(full_years[full_years == full_years.max()].index.max())
answer("q1", "Q1 — year with most additions since 2020", best,
       f"{full_years.max()} additions; ties broken to the most recent year")

# --- structural checks. No published answer key exists for this question. ---
assert 480 <= len(q1) <= 520, f"expected ~500 constituents, got {len(q1)}"
assert q1["year_added"].notna().mean() > 0.90, "too many unparsed dates"
print(f"\n[structural check] {len(q1)} constituents, "
      f"{q1['year_added'].notna().mean():.1%} of dates parsed — plausible.")

# The question's "Additional" prompt.
over_20 = int(((CURRENT_YEAR - q1["year_added"].dropna()) > 20).sum())
print(f"[additional] constituents in the index more than 20 years: {over_20}")

using date column 'Date added', ticker column 'Symbol'

constituents: 503 | year parsed for 503

additions per year from 2020 (2026 excluded as incomplete — it has 13 so far):
year_added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18

ANSWER — Q1 — year with most additions since 2020
  >>> 2025 <<<
  18 additions; ties broken to the most recent year

[structural check] 503 constituents, 100.0% of dates parsed — plausible.
[additional] constituents in the index more than 20 years: 218


---
## Q2 — world indexes YTD vs the S&P 500

> How many indexes (out of 10) have better year-to-date returns than the US
> (S&P 500) as of August 21, 2026?

Eleven tickers are listed for a question phrased "out of 10" — the benchmark is one
of them. The count below therefore excludes `^GSPC` explicitly, which is the trap.

Returns use **Close**, as the question instructs, and are measured from the first to
the last available close inside the window so that a market shut on a boundary date
does not silently drop out.

In [5]:
WORLD_INDEXES = {
    "^GSPC":     "United States — S&P 500",
    "000001.SS": "China — Shanghai Composite",
    "^HSI":      "Hong Kong — Hang Seng",
    "^AXJO":     "Australia — S&P/ASX 200",
    "^NSEI":     "India — Nifty 50",
    "^GSPTSE":   "Canada — S&P/TSX Composite",
    "^GDAXI":    "Germany — DAX",
    "^FTSE":     "United Kingdom — FTSE 100",
    "^N225":     "Japan — Nikkei 225",
    "^MXX":      "Mexico — IPC",
    "^BVSP":     "Brazil — Ibovespa",
}


def compare_returns(tickers, start, end, benchmark="^GSPC", labels=None):
    """Rank period returns on Close and count how many beat the benchmark."""
    data = get_ohlcv(list(tickers), start=start, end=end)
    rows = []
    for t, df in data.items():
        first, last = df["Close"].iloc[0], df["Close"].iloc[-1]
        rows.append({"ticker": t, "name": (labels or {}).get(t, t),
                     "first_date": df.index[0].date(), "last_date": df.index[-1].date(),
                     "return_pct": (last / first - 1) * 100})
    out = pd.DataFrame(rows).sort_values("return_pct", ascending=False).reset_index(drop=True)
    bench = out.loc[out["ticker"] == benchmark, "return_pct"]
    if bench.empty:
        raise ValueError(f"benchmark {benchmark} returned no data")
    return out, int((out["return_pct"] > bench.iloc[0]).sum()), float(bench.iloc[0])


table, n_better, bench_ret = compare_returns(
    WORLD_INDEXES, Q2_START, Q2_END, benchmark="^GSPC", labels=WORLD_INDEXES)

print(f"window {Q2_START} -> {Q2_END} | S&P 500 returned {bench_ret:+.2f}%\n")
print(table.to_string(index=False, float_format=lambda v: f"{v:+.2f}"))
answer("q2", "Q2 — indexes beating the S&P 500", n_better,
       f"as of {Q2_END}, out of {len(table) - 1} non-benchmark indexes")

# The prose used to say "1 January-1 August 2026" while the title and hint said 21
# August. Upstream corrected the prose to 21 August in a6987b0, so this is now a
# sensitivity check on the window rather than a rival reading of the question.
_, n_alt, bench_alt = compare_returns(WORLD_INDEXES, Q2_START, Q2_END_ALT,
                                      benchmark="^GSPC", labels=WORLD_INDEXES)
print(f"\n[sensitivity] with the earlier {Q2_START} -> {Q2_END_ALT} window instead: "
      f"S&P 500 {bench_alt:+.2f}%, {n_alt} indexes ahead."
      + ("  Same answer either way." if n_alt == n_better else
         "  Different — which is why the window wording mattered until it was fixed."))

assert not table["return_pct"].isna().any(), "a NaN return means a broken fetch"
missing = set(WORLD_INDEXES) - set(table["ticker"])
print(f"[structural check] {len(table)}/{len(WORLD_INDEXES)} indexes returned data."
      + (f" Missing: {sorted(missing)}" if missing else ""))

window 2026-01-01 -> 2026-08-21 | S&P 500 returned +11.90%

   ticker                       name first_date  last_date  return_pct
    ^N225         Japan — Nikkei 225 2026-01-05 2026-08-21      +27.36
  ^GSPTSE Canada — S&P/TSX Composite 2026-01-02 2026-08-21      +14.86
    ^GSPC    United States — S&P 500 2026-01-02 2026-08-21      +11.90
    ^FTSE  United Kingdom — FTSE 100 2026-01-02 2026-08-21       +8.70
    ^BVSP          Brazil — Ibovespa 2026-01-02 2026-08-21       +6.54
   ^GDAXI              Germany — DAX 2026-01-02 2026-08-21       +6.51
    ^AXJO    Australia — S&P/ASX 200 2026-01-02 2026-08-21       +3.79
     ^MXX               Mexico — IPC 2026-01-02 2026-08-21       +2.48
     ^HSI      Hong Kong — Hang Seng 2026-01-02 2026-08-21       -1.25
000001.SS China — Shanghai Composite 2026-01-05 2026-08-21       -2.94
    ^NSEI           India — Nifty 50 2026-01-01 2026-08-21       -7.25

ANSWER — Q2 — indexes beating the S&P 500
  >>> 2 <<<
  as of 2026-08-21, out of 10 non

---
## Q3 — corrections from all-time highs

> Calculate the **median drawdown (in %)** of significant market corrections in the
> S&P 500, where a correction is a fall of **at least 5%** from the most recent
> all-time high, measured on closing prices.

Changed from 2025, which asked for median *duration*. Both are computed below; the
drawdown percentile is the submitted answer.

This is the one question with a real numeric self-check: the sheet publishes the top
ten corrections by drawdown, so a correct implementation must reproduce them exactly.

`find_corrections` below is not written in this notebook. It is spliced in from
[`my-notes/lib/corrections.py`](../lib/corrections.py), which is the single
hand-maintained copy and has its own unit tests — including the ten published
corrections asserted against pinned price data, and the `>=` boundary that upstream's
"at least 5%" wording settled.

In [6]:
# Spliced from my-notes/lib/corrections.py — edit there, not here.
# --- BEGIN SHARED: find_corrections ---
CORRECTION_COLUMNS = ["peak_date", "trough_date", "peak", "trough",
                      "drawdown_pct", "duration_days"]


def find_corrections(close: pd.Series, threshold_pct: float = 5.0) -> pd.DataFrame:
    """Drawdown episodes measured from each all-time high.

    Walks consecutive all-time highs; between one ATH and the next, the lowest close
    is the trough. Keeps episodes whose fall reaches `threshold_pct` — inclusive, so
    exactly 5.0% counts, matching the question's "goes down by **at least 5%** from
    the most recent all-time high". Duration is calendar days from ATH to trough —
    peak-to-trough, not peak-to-recovery, which is the convention the published
    table uses.

    An ATH is a close at or above every previous close, so a day that merely matches
    a prior high starts a new episode. Returns one row per episode, and an empty
    frame *with columns* when none qualify.
    """
    close = close.dropna().sort_index()
    ath_dates = list(close.index[close >= close.cummax()])

    episodes = []
    for i, start in enumerate(ath_dates):
        end = ath_dates[i + 1] if i + 1 < len(ath_dates) else close.index[-1]
        window = close.loc[start:end]
        if len(window) < 2:
            continue
        trough_val = window.iloc[1:].min()
        trough_date = window.iloc[1:].idxmin()
        high = close.loc[start]
        dd = (high - trough_val) / high * 100
        if dd >= threshold_pct:
            episodes.append({"peak_date": start.date(), "trough_date": trough_date.date(),
                             "peak": float(high), "trough": float(trough_val),
                             "drawdown_pct": float(dd),
                             "duration_days": int((trough_date - start).days)})
    # columns= matters on the empty path: pd.DataFrame([]) has no columns at all, so
    # a caller reading ["drawdown_pct"] would get KeyError rather than an empty Series.
    return pd.DataFrame(episodes, columns=CORRECTION_COLUMNS)
# --- END SHARED ---


spx = get_ohlcv("^GSPC", start=HISTORY_START)["^GSPC"]
print(f"^GSPC: {len(spx)} bars, {spx.index[0].date()} -> {spx.index[-1].date()}")

corrections = find_corrections(spx["Close"], threshold_pct=5.0)

dd25, dd50, dd75 = corrections["drawdown_pct"].quantile([0.25, 0.50, 0.75])
du25, du50, du75 = corrections["duration_days"].quantile([0.25, 0.50, 0.75])

print(f"\ncorrections >= 5%: {len(corrections)}")
print(f"drawdown %  — 25th {dd25:.2f} | median {dd50:.2f} | 75th {dd75:.2f}")
print(f"duration d  — 25th {du25:.0f} | median {du50:.0f} | 75th {du75:.0f}")

answer("q3", "Q3 — median drawdown of corrections (%)", f"{dd50:.2f}%")
print(f"  (2025 asked for duration instead; that median is {du50:.0f} days)")

^GSPC: 19284 bars, 1950-01-03 -> 2026-08-26

corrections >= 5%: 74
drawdown %  — 25th 6.23 | median 7.99 | 75th 14.02
duration d  — 25th 22 | median 40 | 75th 86

ANSWER — Q3 — median drawdown of corrections (%)
  >>> 7.99% <<<
  (2025 asked for duration instead; that median is 40 days)


In [7]:
# --- numeric self-check, against the table printed in the question itself ---
PUBLISHED_TOP10 = [
    ("2007-10-09", "2009-03-09", 56.8, 517), ("2000-03-24", "2002-10-09", 49.1, 929),
    ("1973-01-11", "1974-10-03", 48.2, 630), ("1968-11-29", "1970-05-26", 36.1, 543),
    ("2020-02-19", "2020-03-23", 33.9,  33), ("1987-08-25", "1987-12-04", 33.5, 101),
    ("1961-12-12", "1962-06-26", 28.0, 196), ("1980-11-28", "1982-08-12", 27.1, 622),
    ("2022-01-03", "2022-10-12", 25.4, 282), ("1966-02-09", "1966-10-07", 22.2, 240),
]

top10 = corrections.nlargest(10, "drawdown_pct").reset_index(drop=True)
print("computed top 10 by drawdown:")
print(top10.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

print("\nvs published:")
ok = 0
for i, (pk, tr, dd, dur) in enumerate(PUBLISHED_TOP10):
    if i >= len(top10):
        print(f"  {pk}: MISSING from computed top-10")
        continue
    r = top10.loc[i]
    hit = (str(r["peak_date"]) == pk and str(r["trough_date"]) == tr
           and abs(r["drawdown_pct"] - dd) < 0.5 and abs(r["duration_days"] - dur) <= 2)
    ok += hit
    print(f"  {'OK  ' if hit else 'DIFF'} {pk} -> {tr}  {dd}%  {dur}d"
          f"   | computed {r['peak_date']} -> {r['trough_date']} "
          f"{r['drawdown_pct']:.1f}% {r['duration_days']}d")

print(f"\n[numeric check] {ok}/10 published corrections reproduced exactly.")
if ok < 8:
    print("  NOTE: under 8 matches means the ATH/trough definition differs from the "
          "instructor's. Check the 5% threshold and the peak-to-trough (not "
          "peak-to-recovery) duration convention before submitting.")

computed top 10 by drawdown:
 peak_date trough_date    peak  trough  drawdown_pct  duration_days
2007-10-09  2009-03-09 1565.15  676.53         56.78            517
2000-03-24  2002-10-09 1527.46  776.76         49.15            929
1973-01-11  1974-10-03  120.24   62.28         48.20            630
1968-11-29  1970-05-26  108.37   69.29         36.06            543
2020-02-19  2020-03-23 3386.15 2237.40         33.92             33
1987-08-25  1987-12-04  336.77  223.92         33.51            101
1961-12-12  1962-06-26   72.64   52.32         27.97            196
1980-11-28  1982-08-12  140.52  102.42         27.11            622
2022-01-03  2022-10-12 4796.56 3577.03         25.43            282
1966-02-09  1966-10-07   94.06   73.20         22.18            240

vs published:
  OK   2007-10-09 -> 2009-03-09  56.8%  517d   | computed 2007-10-09 -> 2009-03-09 56.8% 517d
  OK   2000-03-24 -> 2002-10-09  49.1%  929d   | computed 2000-03-24 -> 2002-10-09 49.1% 929d
  OK   1973-01-11 ->

---
## Q4 — AMZN earnings surprises

> Load earnings data with `get_earnings_dates()`, compute the 2-day return as
> `Close_Day3 / Close_Day1 - 1` around each announcement, then **filter for positive
> earnings surprises and calculate the median 2-day return** — and after that, the
> correlation between the 2-day return and the surprise magnitude.

Step 4 originally asked only for "the correlation of a stock return vs. earnings
surprise", which contradicted this question's own heading. Upstream rewrote it in
`a5cf53a` to ask for the **median first** and the correlation as a follow-up, so the
median is the submitted answer and the correlation is printed beneath it.

Upstream also now states the expected shape — **25 entries starting 2020-10-29**, one
of them a future date with no reported EPS — which is exactly the 24 matched
announcements below.

Two mechanical details:

- `get_earnings_dates()` returns a **tz-aware** index (America/New_York) stamped
  16:00 — after the close. Comparing that to naive timestamps raises, so it is
  localised away before matching.
- Because the release is after the close, the announcement date is Day 2 and the
  reaction lands on Day 3. `Close_Day3 / Close_Day1 - 1` centred on the announcement
  day captures it, which is exactly what the question specifies.

In [8]:
def two_day_returns(close):
    """Close_Day3 / Close_Day1 - 1, indexed by Day 2 (the announcement day)."""
    return pd.Series(close.shift(-1).values / close.shift(1).values,
                     index=close.index, name="ret_2d") - 1


ticker_obj = yf.Ticker(Q4_TICKER)
earnings = ticker_obj.get_earnings_dates()
print(f"get_earnings_dates(): {earnings.shape} | columns {list(earnings.columns)}")

e = earnings.reset_index()
e.columns = ["earnings_date", "eps_estimate", "eps_actual", "surprise_pct"]

# tz-aware (America/New_York, 16:00) -> naive calendar day, so it can be matched
# against the price index without raising.
e["date"] = pd.to_datetime(e["earnings_date"]).dt.tz_localize(None).dt.normalize()
e = e.dropna(subset=["eps_actual", "surprise_pct"])      # drop the unreported quarter
print(f"reported quarters: {len(e)}  ({e['date'].min().date()} -> {e['date'].max().date()})")

px = get_ohlcv(Q4_TICKER, start=HISTORY_START)[Q4_TICKER]
ret2d = two_day_returns(px["Close"]).dropna()
print(f"{Q4_TICKER}: {len(px)} bars, 2-day returns computed for {len(ret2d)}")

# Snap each announcement to its trading day (or the next one, if it fell on a holiday).
trading_days = pd.DatetimeIndex(ret2d.index)
pos = trading_days.searchsorted(e["date"].values)
keep = pos < len(trading_days)
e = e.loc[keep].copy()
e["match_date"] = trading_days[pos[keep]]
e["ret_2d_pct"] = ret2d.reindex(e["match_date"]).to_numpy() * 100
e = e.dropna(subset=["ret_2d_pct"])

print(f"\nmatched announcements: {len(e)}")
print(e[["date", "eps_estimate", "eps_actual", "surprise_pct", "ret_2d_pct"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

get_earnings_dates(): (25, 3) | columns ['EPS Estimate', 'Reported EPS', 'Surprise(%)']
reported quarters: 24  (2020-10-29 -> 2026-07-30)


AMZN: 7366 bars, 2-day returns computed for 7364

matched announcements: 24
      date  eps_estimate  eps_actual  surprise_pct  ret_2d_pct
2026-07-30          1.83        5.75        215.02       19.82
2026-04-29          1.64        2.78         69.02        2.06
2026-02-05          1.95        1.95          0.22       -9.73
2025-10-30          1.56        1.95         25.20        6.04
2025-07-31          1.32        1.68         27.19       -6.71
2025-05-01          1.36        1.59         16.77        3.01
2025-02-06          1.48        1.86         25.29       -2.97
2024-10-31          1.14        1.43         25.22        2.70
2024-08-01          1.02        1.26         23.76      -10.20
2024-04-30          0.83        0.98         17.67       -1.08
2024-02-01          0.80        1.00         24.38       10.70
2023-10-26          0.58        0.94         62.22        5.23
2023-08-03          0.34        0.65         91.19        8.86
2023-04-27          0.21        0.31      

In [9]:
positive = e[e["surprise_pct"] > 0]
answer("q4", "Q4 — median 2-day return after a positive surprise",
       f"{positive['ret_2d_pct'].median():.2f}%",
       f"n = {len(positive)} positive surprises out of {len(e)} announcements")

# Step 4 asks for the correlation immediately after the median, so it is reported
# too — just not as the graded scalar.
corr = e["ret_2d_pct"].corr(e["surprise_pct"])
ANSWERS["q4_corr"] = f"{corr:.4f}"
print(f"\n[also asked] correlation of 2-day return vs surprise: {corr:.4f} "
      f"(Pearson, n = {len(e)})")
print(f"[baseline]   median 2-day return over all history: "
      f"{ret2d.median() * 100:.2f}%")

# Spearman too: a single enormous surprise dominates a Pearson correlation on 24 points.
print(f"\n[robustness] Spearman rank correlation: "
      f"{e['ret_2d_pct'].corr(e['surprise_pct'], method='spearman'):.4f}")
print(f"             largest surprise in the sample: {e['surprise_pct'].max():.1f}%")

assert len(e) >= 20, f"only {len(e)} matched announcements — expected ~24"
print(f"\n[structural check] {len(e)} matched announcements, no NaNs in either series.")


ANSWER — Q4 — median 2-day return after a positive surprise
  >>> 0.35% <<<
  n = 20 positive surprises out of 24 announcements

[also asked] correlation of 2-day return vs surprise: 0.2191 (Pearson, n = 24)
[baseline]   median 2-day return over all history: 0.16%



[robustness] Spearman rank correlation: 0.2835
             largest surprise in the sample: 657.1%

[structural check] 24 matched announcements, no NaNs in either series.


---
## Q5 — capstone idea (free text, optional)

**A short-horizon recommendation system for US large caps, built on the lakehouse
this coursework already stands up, with an LLM layer that can only argue from
features that actually exist.**

Concretely:

- **Universe and horizon.** The 190 US large caps already in
  `bronze.ohlcv_daily`, on a 1–4 week horizon. Not a new universe — a deeper one.
- **Close the two gaps this homework exposed.** Q1 needed S&P 500 membership
  history and Q4 needed an earnings calendar; neither is in bronze, which is
  exactly why `crosscheck_bronze` can only answer two of the four questions.
  Ingesting both is the first capstone task, and it is motivated by evidence
  rather than guessed at. Index-addition dates are independently interesting: the
  question's own context notes that new entrants pop on announcement.
- **Features.** TA-Lib indicators from Module 2, plus macro regime features from
  the 17 FRED series already loaded — yield-curve slope (`DGS10 - DGS2`), credit
  spread (`BAA - AAA`), breakeven inflation (`T10YIE`), `VIXCLS`. The macro side
  is free; it is already sitting in `bronze.macro_series`.
- **Model.** sklearn on a strictly temporal split, with the split boundaries
  written to `ml.model_runs` on every run — the schema already has columns for
  them precisely because lookahead bias is the easiest mistake to make here.
- **Simulation.** `sim.trades` with `fees` never null. Fees are what kill
  high-frequency strategies, and a good model that loses money after costs is the
  normal outcome, not the surprising one.
- **The AI layer, and its constraint.** An agent that turns each prediction into a
  written rationale — but restricted to citing features that exist in the feature
  store, so it cannot invent a reason the model did not use. The interesting
  problem is not generating the text, it is making the explanation *faithful* to
  the model. That constraint is the project.

Risk I already know about: TensorFlow will not import on Databricks serverless
(protobuf conflict), so any deep-learning component runs in Colab or locally.

## Q6 — additional metrics (free text, optional)

Seventeen macro series are already loaded, so this question is a `GROUP BY` rather
than a download:

```sql
SELECT series_id, count(*), min(date), max(date)
FROM stock_analytics.bronze.macro_series GROUP BY 1 ORDER BY 1;
```

**Already in, and why each earns its place:**

| Series | Why it matters |
|---|---|
| `DGS1 DGS2 DGS3 DGS5 DGS10 DGS30` | The whole curve, so slope and inversion are derivable rather than assumed. Inversion has preceded every recent US recession |
| `AAA`, `BAA` | Their spread is a clean risk-appetite proxy that widens before equity drawdowns |
| `T10YIE` | Market-implied inflation — separates a nominal rate move from a real one |
| `VIXCLS`, `GVZCLS` | Equity and gold implied volatility; regime labels, and VIX is mean-reverting enough to be a feature rather than noise |
| `FEDFUNDS`, `CPILFESL`, `GDPPOT` | The policy triangle the lectures build up from |
| `DCOILWTICO`, `DCOILBRENTEU` | Input costs; their spread is a transport/logistics signal |

**Worth adding next, in priority order:**

1. **S&P 500 membership history** — the add/drop dates Q1 scrapes from Wikipedia.
   Storing them turns a one-off scrape into a joinable dimension and makes index
   -addition events a tradeable feature.
2. **Earnings calendar with surprise** — `get_earnings_dates()` per ticker, which
   Q4 needs. Note the trap Ivan flags: financial-statement data arrives 1–2 months
   after quarter close, so joining on report date leaks the future into training.
   Store both the announcement date and the period it covers.
3. **Short interest** and **sector-ETF relative strength** — crowding and rotation,
   neither derivable from OHLCV alone.

Alpha Vantage is connected via MCP and covers most of this, but its free tier is
**25 requests/day** — a fallback for specific symbols, not a bulk source. Anything
universe-wide needs a different provider or a lot of patience.

---
## Submission block

In [10]:
import json

print(json.dumps(ANSWERS, indent=2))

# On Databricks this returns the answers through the Jobs API, which discards
# print() output — so the same notebook works as a scheduled job. Off Databricks
# `dbutils` is simply not defined and this is a no-op. It is deliberately the
# very last statement: notebook.exit() stops execution, so anything after it
# would never run interactively.
try:
    dbutils.notebook.exit(json.dumps(ANSWERS))   # noqa: F821
except NameError:
    pass

{
  "q1": 2025,
  "q2": 2,
  "q3": "7.99%",
  "q4": "0.35%",
  "q4_corr": "0.2191"
}
